01. Calculo DAN con Reporte DANES

In [1]:
# ==================== PROCESAMIENTO COMPLETO DE ARCHIVOS DAN ====================
import pandas as pd
import os
import glob
from datetime import datetime

# Configuración de rutas
carpeta_dan = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Calculo de PC y DAN\Abastos\DAN'
carpeta_outputs = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Calculo de PC y DAN\Abastos\Outputs'

# Crear carpeta de outputs si no existe
os.makedirs(carpeta_outputs, exist_ok=True)

# PASO 1: EXPLORACIÓN DE LA CARPETA DAN
print("🔍 EXPLORANDO CARPETA DAN...")
print("="*60)

if os.path.exists(carpeta_dan):
    print(f"Contenido de la carpeta: {carpeta_dan}")
    print("-" * 50)
    
    # Listar todos los archivos en la carpeta
    archivos = os.listdir(carpeta_dan)
    
    if archivos:
        for i, archivo in enumerate(archivos, 1):
            ruta_completa = os.path.join(carpeta_dan, archivo)
            tamaño = os.path.getsize(ruta_completa) / 1024  # Tamaño en KB
            print(f"{i}. {archivo} ({tamaño:.1f} KB)")
            
        print(f"\nTotal de archivos encontrados: {len(archivos)}")
        
        # Buscar archivos Excel y CSV específicamente
        archivos_excel = glob.glob(os.path.join(carpeta_dan, '*.xlsx')) + glob.glob(os.path.join(carpeta_dan, '*.xlsm')) + glob.glob(os.path.join(carpeta_dan, '*.xls'))
        archivos_csv = glob.glob(os.path.join(carpeta_dan, '*.csv'))
        
        if archivos_excel:
            print(f"\nArchivos Excel encontrados: {len(archivos_excel)}")
            for excel_file in archivos_excel:
                print(f"- {os.path.basename(excel_file)}")
        
        if archivos_csv:
            print(f"\nArchivos CSV encontrados: {len(archivos_csv)}")
            for csv_file in archivos_csv:
                print(f"- {os.path.basename(csv_file)}")
                
        if not archivos_excel and not archivos_csv:
            print("\nNo se encontraron archivos Excel ni CSV en la carpeta.")
            
    else:
        print("La carpeta está vacía.")
        
else:
    print(f"La carpeta no existe: {carpeta_dan}")

print("\n" + "="*60)

# PASO 2: PROCESAMIENTO AUTOMÁTICO DE ARCHIVOS
print("\n🔍 PROCESANDO ARCHIVOS DAN (Excel y CSV)...")
print("="*60)

# Inicializar lista para almacenar todos los DataFrames
list_df_danes = []

# Buscar todos los archivos Excel y CSV en la carpeta DAN
archivos_excel = glob.glob(os.path.join(carpeta_dan, '*.xlsx')) + \
                 glob.glob(os.path.join(carpeta_dan, '*.xlsm')) + \
                 glob.glob(os.path.join(carpeta_dan, '*.xls'))
archivos_csv = glob.glob(os.path.join(carpeta_dan, '*.csv'))

# Combinar todos los archivos
todos_archivos = archivos_excel + archivos_csv

if todos_archivos:
    for archivo in todos_archivos:
        nombre_archivo = os.path.basename(archivo)
        extension = os.path.splitext(archivo)[1].lower()
        print(f"\n📂 Procesando: {nombre_archivo} (Tipo: {extension})")
        
        try:
            if extension == '.csv':
                # Procesar archivo CSV
                try:
                    # Intentar leer con diferentes encodings
                    encodings = ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']
                    df_temp = None
                    
                    for encoding in encodings:
                        try:
                            df_temp = pd.read_csv(archivo, encoding=encoding)
                            print(f"   ✅ CSV leído exitosamente con encoding: {encoding}")
                            break
                        except UnicodeDecodeError:
                            continue
                    
                    if df_temp is None:
                        print(f"   ❌ No se pudo leer el CSV con ningún encoding probado")
                        continue
                    
                    if not df_temp.empty:
                        # Agregar columnas de metadatos para identificar origen
                        df_temp['ARCHIVO_ORIGEN'] = nombre_archivo
                        df_temp['HOJA_ORIGEN'] = 'CSV_DATA'
                        
                        # Estandarizar nombres de columnas
                        df_temp.columns = df_temp.columns.str.upper().str.strip()
                        
                        # Agregar a la lista
                        list_df_danes.append(df_temp)
                        print(f"   ✅ CSV procesado: {df_temp.shape[0]} filas")
                    else:
                        print(f"   ⚠️ CSV vacío, omitido")
                        
                except Exception as e:
                    print(f"   ❌ Error procesando CSV: {e}")
                    
            else:
                # Procesar archivo Excel
                excel_file = pd.ExcelFile(archivo)
                print(f"   Hojas encontradas: {len(excel_file.sheet_names)} -> {excel_file.sheet_names}")
                
                # Procesar cada hoja del archivo
                for hoja in excel_file.sheet_names:
                    try:
                        df_temp = pd.read_excel(archivo, sheet_name=hoja)
                        
                        if not df_temp.empty:
                            # Agregar columnas de metadatos para identificar origen
                            df_temp['ARCHIVO_ORIGEN'] = nombre_archivo
                            df_temp['HOJA_ORIGEN'] = hoja
                            
                            # Estandarizar nombres de columnas
                            df_temp.columns = df_temp.columns.str.upper().str.strip()
                            
                            # Agregar a la lista
                            list_df_danes.append(df_temp)
                            print(f"   ✅ Hoja '{hoja}': {df_temp.shape[0]} filas procesadas")
                        else:
                            print(f"   ⚠️ Hoja '{hoja}': vacía, omitida")
                            
                    except Exception as e:
                        print(f"   ❌ Error procesando hoja '{hoja}': {e}")
                        
        except Exception as e:
            print(f"❌ Error procesando archivo '{nombre_archivo}': {e}")

    # PASO 3: CONSOLIDACIÓN Y TRANSFORMACIÓN DE DATOS
    if list_df_danes:
        df_danes = pd.concat(list_df_danes, ignore_index=True)
        
        # Crear la columna CONCA concatenando RUTA y CLAVEPRODUCTO
        if 'CLAVEDOCUMENTO' in df_danes.columns and 'CLAVEPRODUCTO' in df_danes.columns:
            import re
            
            def extraer_ruta(clavedocumento):
                if pd.isna(clavedocumento):
                    return None
                # Extraer solo los números de CLAVEDOCUMENTO para crear RUTA
                numeros = re.findall(r'\d+', str(clavedocumento))
                if numeros:
                    # Tomar el primer grupo de números encontrado
                    return numeros[0]
                return None
            
            def crear_conca(row):
                ruta = extraer_ruta(row['CLAVEDOCUMENTO'])
                claveproducto = row['CLAVEPRODUCTO']
                
                if pd.isna(ruta) or pd.isna(claveproducto):
                    return None
                # Concatenar RUTA + CLAVEPRODUCTO
                return f"{ruta}{claveproducto}"
            
            # Crear columna RUTA
            df_danes['RUTA'] = df_danes['CLAVEDOCUMENTO'].apply(extraer_ruta)
            
            # Crear columna CONCA
            df_danes['CONCA'] = df_danes.apply(crear_conca, axis=1)
            
            # ESTANDARIZAR TIPO DE DATOS Y LIMPIAR ESPACIOS
            df_danes['CONCA'] = df_danes['CONCA'].astype(str).str.strip().str.replace('.0', '', regex=False)
            
            print(f"\n✅ Columnas RUTA y CONCA creadas exitosamente")
            print(f"   - CONCA estandarizada como string sin espacios ni decimales")
            
            # Mostrar algunos ejemplos de la transformación
            ejemplos = df_danes[['CLAVEDOCUMENTO', 'RUTA', 'CLAVEPRODUCTO', 'CONCA']].dropna().head(3)
            if not ejemplos.empty:
                print("📝 Ejemplos de transformación:")
                for _, row in ejemplos.iterrows():
                    print(f"   {row['CLAVEDOCUMENTO']} → RUTA: {row['RUTA']} + PRODUCTO: {row['CLAVEPRODUCTO']} = CONCA: {row['CONCA']}")
        else:
            print("⚠️ No se encontraron las columnas CLAVEDOCUMENTO o CLAVEPRODUCTO para crear CONCA")
        
        # PASO 4: AGRUPACIÓN POR CONCA
        if 'CONCA' in df_danes.columns and 'CANTIDAD' in df_danes.columns:
            print(f"\n📊 AGRUPANDO POR CONCA...")
            print("-" * 40)
            
            # Crear DataFrame agrupado
            df_danes_agrupado = df_danes.groupby('CONCA').agg({
                'CANTIDAD': 'sum',
                'CLAVEDOCUMENTO': 'first',
                'RUTA': 'first',
                'CLAVEPRODUCTO': 'first'
            }).reset_index()
            
            print(f"✅ Agrupación completada:")
            print(f"   Registros originales: {df_danes.shape[0]}")
            print(f"   Registros agrupados: {df_danes_agrupado.shape[0]}")
            print(f"   Reducción: {df_danes.shape[0] - df_danes_agrupado.shape[0]} registros consolidados")
            
            # Mostrar primeros resultados agrupados
            print(f"\n📝 Primeros 5 registros agrupados:")
            print(df_danes_agrupado.head().to_string())
            
            # Mostrar estadísticas de cantidad
            print(f"\n📈 Estadísticas de CANTIDAD agrupada:")
            print(f"   Total cantidad: {df_danes_agrupado['CANTIDAD'].sum():,.0f}")
            print(f"   Cantidad promedio: {df_danes_agrupado['CANTIDAD'].mean():.2f}")
            print(f"   Cantidad máxima: {df_danes_agrupado['CANTIDAD'].max():,.0f}")
            print(f"   Cantidad mínima: {df_danes_agrupado['CANTIDAD'].min():,.0f}")
            
        else:
            print("⚠️ No se pueden agrupar los datos: faltan columnas CONCA o CANTIDAD")
            df_danes_agrupado = None
        
        # PASO 5: RESUMEN DE RESULTADOS
        print(f"\n📊 RESULTADO CONSOLIDADO:")
        print("="*60)
        print(f"Total de registros: {df_danes.shape[0]}")
        print(f"Total de columnas: {df_danes.shape[1]}")
        
        print(f"\nColumnas en el DataFrame final:")
        for i, col in enumerate(df_danes.columns, 1):
            print(f"  {i}. {col}")
        
        # Mostrar resumen por origen
        print(f"\nResumen por archivo/hoja:")
        resumen_origen = df_danes.groupby(['ARCHIVO_ORIGEN', 'HOJA_ORIGEN']).size().reset_index(name='REGISTROS')
        print(resumen_origen.to_string(index=False))
        
        # Mostrar primeras filas
        print(f"\nPrimeras 5 filas del DataFrame consolidado:")
        print(df_danes.head().to_string())
        
        # PASO 6: DATOS PREPARADOS PARA EXPORTACIÓN FINAL
        print(f"\n✅ DATOS DAN PREPARADOS:")
        print(f"📋 df_danes_agrupado: {df_danes_agrupado.shape[0] if df_danes_agrupado is not None else 0} registros")
        print(f"📄 Será incluido en el archivo Excel final consolidado")
    
    else:
        print("⚠️ No se pudieron procesar datos de ningún archivo.")
        
else:
    print("⚠️ No se encontraron archivos Excel ni CSV en la carpeta DAN.")

print(f"\n🎉 PROCESAMIENTO COMPLETADO!")
print("="*60)

🔍 EXPLORANDO CARPETA DAN...
Contenido de la carpeta: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Calculo de PC y DAN\Abastos\DAN
--------------------------------------------------
1. Reporte de Devoluciones.xlsx (11.2 KB)

Total de archivos encontrados: 1

Archivos Excel encontrados: 1
- Reporte de Devoluciones.xlsx


🔍 PROCESANDO ARCHIVOS DAN (Excel y CSV)...

📂 Procesando: Reporte de Devoluciones.xlsx (Tipo: .xlsx)
   Hojas encontradas: 1 -> ['Exportar Hoja de Trabajo']
   ✅ Hoja 'Exportar Hoja de Trabajo': 270 filas procesadas

✅ Columnas RUTA y CONCA creadas exitosamente
   - CONCA estandarizada como string sin espacios ni decimales
📝 Ejemplos de transformación:
   DAN72048             → RUTA: 72048 + PRODUCTO: 16028 = CONCA: 7204816028
   DAN72048             → RUTA: 72048 + PRODUCTO: 19963 = CONCA: 7204819963
   DAN72028             → RUTA: 72028 + PRODUCTO: 29943 = CONCA: 7202829943

📊 AGRUPANDO POR CONCA...
----------------------------------------
✅ Ag

c:\Users\igcamposg\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


02. Calculo DAN con PC

In [2]:
# ==================== PROCESAMIENTO COMPLETO DE ARCHIVOS PC ====================
import pandas as pd
import os
import glob
from datetime import datetime

# Configuración de rutas
carpeta_pc = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Calculo de PC y DAN\Abastos\PC'
carpeta_outputs = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Calculo de PC y DAN\Abastos\Outputs'

# Crear carpeta de outputs si no existe
os.makedirs(carpeta_outputs, exist_ok=True)

# PASO 1: EXPLORACIÓN DE LA CARPETA PC
print("🔍 EXPLORANDO CARPETA PC...")
print("="*60)

if os.path.exists(carpeta_pc):
    print(f"Contenido de la carpeta: {carpeta_pc}")
    print("-" * 50)
    
    # Listar todos los archivos en la carpeta
    archivos = os.listdir(carpeta_pc)
    
    if archivos:
        for i, archivo in enumerate(archivos, 1):
            ruta_completa = os.path.join(carpeta_pc, archivo)
            tamaño = os.path.getsize(ruta_completa) / 1024  # Tamaño en KB
            print(f"{i}. {archivo} ({tamaño:.1f} KB)")
            
        print(f"\nTotal de archivos encontrados: {len(archivos)}")
        
        # Buscar archivos Excel y CSV específicamente
        archivos_excel = glob.glob(os.path.join(carpeta_pc, '*.xlsx')) + glob.glob(os.path.join(carpeta_pc, '*.xlsm')) + glob.glob(os.path.join(carpeta_pc, '*.xls'))
        archivos_csv = glob.glob(os.path.join(carpeta_pc, '*.csv'))
        
        if archivos_excel:
            print(f"\nArchivos Excel encontrados: {len(archivos_excel)}")
            for excel_file in archivos_excel:
                print(f"- {os.path.basename(excel_file)}")
        
        if archivos_csv:
            print(f"\nArchivos CSV encontrados: {len(archivos_csv)}")
            for csv_file in archivos_csv:
                print(f"- {os.path.basename(csv_file)}")
                
        if not archivos_excel and not archivos_csv:
            print("\nNo se encontraron archivos Excel ni CSV en la carpeta.")
            
    else:
        print("La carpeta está vacía.")
        
else:
    print(f"La carpeta no existe: {carpeta_pc}")

print("\n" + "="*60)

# PASO 2: PROCESAMIENTO AUTOMÁTICO DE ARCHIVOS
print("\n🔍 PROCESANDO ARCHIVOS PC (Excel y CSV)...")
print("="*60)

# Inicializar lista para almacenar todos los DataFrames
list_df_pc = []

# Buscar todos los archivos Excel y CSV en la carpeta PC
archivos_excel = glob.glob(os.path.join(carpeta_pc, '*.xlsx')) + \
                 glob.glob(os.path.join(carpeta_pc, '*.xlsm')) + \
                 glob.glob(os.path.join(carpeta_pc, '*.xls'))
archivos_csv = glob.glob(os.path.join(carpeta_pc, '*.csv'))

# Combinar todos los archivos
todos_archivos = archivos_excel + archivos_csv

if todos_archivos:
    for archivo in todos_archivos:
        nombre_archivo = os.path.basename(archivo)
        extension = os.path.splitext(archivo)[1].lower()
        print(f"\n📂 Procesando: {nombre_archivo} (Tipo: {extension})")
        
        try:
            if extension == '.csv':
                # Procesar archivo CSV
                try:
                    # Intentar leer con diferentes encodings
                    encodings = ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']
                    df_temp = None
                    
                    for encoding in encodings:
                        try:
                            df_temp = pd.read_csv(archivo, encoding=encoding)
                            print(f"   ✅ CSV leído exitosamente con encoding: {encoding}")
                            break
                        except UnicodeDecodeError:
                            continue
                    
                    if df_temp is None:
                        print(f"   ❌ No se pudo leer el CSV con ningún encoding probado")
                        continue
                    
                    if not df_temp.empty:
                        # Agregar columnas de metadatos para identificar origen
                        df_temp['ARCHIVO_ORIGEN'] = nombre_archivo
                        df_temp['HOJA_ORIGEN'] = 'CSV_DATA'
                        
                        # Estandarizar nombres de columnas
                        df_temp.columns = df_temp.columns.str.upper().str.strip()
                        
                        # Agregar a la lista
                        list_df_pc.append(df_temp)
                        print(f"   ✅ CSV procesado: {df_temp.shape[0]} filas")
                    else:
                        print(f"   ⚠️ CSV vacío, omitido")
                        
                except Exception as e:
                    print(f"   ❌ Error procesando CSV: {e}")
                    
            else:
                # Procesar archivo Excel
                excel_file = pd.ExcelFile(archivo)
                print(f"   Hojas encontradas: {len(excel_file.sheet_names)} -> {excel_file.sheet_names}")
                
                # Procesar cada hoja del archivo
                for hoja in excel_file.sheet_names:
                    try:
                        df_temp = pd.read_excel(archivo, sheet_name=hoja)
                        
                        if not df_temp.empty:
                            # Agregar columnas de metadatos para identificar origen
                            df_temp['ARCHIVO_ORIGEN'] = nombre_archivo
                            df_temp['HOJA_ORIGEN'] = hoja
                            
                            # Estandarizar nombres de columnas
                            df_temp.columns = df_temp.columns.str.upper().str.strip()
                            
                            # Agregar a la lista
                            list_df_pc.append(df_temp)
                            print(f"   ✅ Hoja '{hoja}': {df_temp.shape[0]} filas procesadas")
                        else:
                            print(f"   ⚠️ Hoja '{hoja}': vacía, omitida")
                            
                    except Exception as e:
                        print(f"   ❌ Error procesando hoja '{hoja}': {e}")
                        
        except Exception as e:
            print(f"❌ Error procesando archivo '{nombre_archivo}': {e}")

    # PASO 3: CONSOLIDACIÓN Y TRANSFORMACIÓN DE DATOS
    if list_df_pc:
        df_pc = pd.concat(list_df_pc, ignore_index=True)
        
        # Crear la columna CONCA usando LOTETERMINADO y TRIM(DE.CLAVEPRODUCTO)
        # Buscar las columnas específicas de PC (pueden tener diferentes nombres)
        columna_lote = None
        columna_producto = None
        
        # Identificar las columnas correctas
        for col in df_pc.columns:
            if 'LOTETERMINADO' in col.upper():
                columna_lote = col
            elif col.upper() == 'TRIM(DE.CLAVEPRODUCTO)':
                columna_producto = col
        
        # Si no encuentra la columna exacta, buscar una que contenga solo TRIM(DE.CLAVEPRODUCTO)
        if columna_producto is None:
            for col in df_pc.columns:
                if 'TRIM(DE.CLAVEPRODUCTO)' in col.upper() and 'SELECT' not in col.upper():
                    columna_producto = col
                    break
        
        if columna_lote is not None and columna_producto is not None:
            print(f"🔍 Columnas identificadas:")
            print(f"   - Lote: {columna_lote}")
            print(f"   - Producto: {columna_producto}")
            
            def extraer_ruta_pc(loteterminado):
                if pd.isna(loteterminado):
                    return None
                # Extraer los últimos 5 dígitos de LOTETERMINADO
                lote_str = str(loteterminado).strip()
                if len(lote_str) >= 5:
                    return lote_str[-5:]  # Últimos 5 caracteres
                else:
                    return lote_str  # Si tiene menos de 5 caracteres, devolver todo
            
            def crear_conca_pc(row):
                ruta = extraer_ruta_pc(row[columna_lote])
                claveproducto = row[columna_producto]
                
                if pd.isna(ruta) or pd.isna(claveproducto):
                    return None
                # Concatenar RUTA + TRIM(DE.CLAVEPRODUCTO)
                return f"{ruta}{claveproducto}"
            
            # Crear columna RUTA
            df_pc['RUTA'] = df_pc[columna_lote].apply(extraer_ruta_pc)
            
            # Crear columna CONCA
            df_pc['CONCA'] = df_pc.apply(crear_conca_pc, axis=1)
            
            # ESTANDARIZAR TIPO DE DATOS Y LIMPIAR ESPACIOS
            df_pc['CONCA'] = df_pc['CONCA'].astype(str).str.strip()
            
            print(f"\n✅ Columnas RUTA y CONCA creadas exitosamente")
            print(f"   - CONCA estandarizada como string sin espacios")
            
            # Crear columnas de clasificación por tipo de material
            # Definir los códigos de productos por categoría
            codigos_calentadores = {45270, 45272, 45273, 45274, 49965, 49966}
            codigos_escaleras = {
                16740, 16763, 24117, 24118, 24119, 24120, 100495, 101883, 101903, 101904, 
                101905, 101906, 10264, 10435, 10438, 10445, 10495, 16741, 16742, 16764, 
                16765, 24122, 100224, 101884, 10335, 10437, 10439, 10442, 10450, 10459, 
                10496, 10497, 16743, 16744, 16746, 16747, 100225, 100226, 10337, 10338, 
                10436, 10460, 10469, 10498, 16026, 16027, 16028, 16748, 16757, 16758, 
                16759, 100227, 100228, 100229
            }
            
            def clasificar_material(claveproducto):
                if pd.isna(claveproducto):
                    return 0, 0  # (calentador, escalera)
                try:
                    codigo = int(float(str(claveproducto).strip()))
                    calentador = 1 if codigo in codigos_calentadores else 0
                    escalera = 1 if codigo in codigos_escaleras else 0
                    return calentador, escalera
                except (ValueError, TypeError):
                    return 0, 0
            
            # Aplicar clasificación
            clasificacion = df_pc[columna_producto].apply(clasificar_material)
            df_pc['Calentadores'] = [x[0] for x in clasificacion]
            df_pc['Escaleras'] = [x[1] for x in clasificacion]
            
            print(f"✅ Columnas de clasificación creadas: Calentadores y Escaleras")
            
            # Crear columna DAN+OTROS (diferencia entre CANTIDADASIGNADA y CANTIDADCARGADA)
            # Buscar las columnas de cantidad
            columna_asignada = None
            columna_cargada = None
            
            for col in df_pc.columns:
                if 'CANTIDADASIGNADA' in col.upper():
                    columna_asignada = col
                elif 'CANTIDADCARGADA' in col.upper():
                    columna_cargada = col
            
            if columna_asignada is not None and columna_cargada is not None:
                print(f"🔍 Columnas de cantidad identificadas:")
                print(f"   - Cantidad Asignada: {columna_asignada}")
                print(f"   - Cantidad Cargada: {columna_cargada}")
                
                # Crear columna DAN+OTROS
                df_pc['DAN+OTROS'] = df_pc[columna_asignada] - df_pc[columna_cargada]
                
                print(f"✅ Columna DAN+OTROS creada (CANTIDADASIGNADA - CANTIDADCARGADA)")
                
                # Mostrar estadísticas de DAN+OTROS
                print(f"📊 Estadísticas de DAN+OTROS:")
                print(f"   - Total DAN+OTROS: {df_pc['DAN+OTROS'].sum():,.0f}")
                print(f"   - Promedio: {df_pc['DAN+OTROS'].mean():.2f}")
                print(f"   - Valores positivos: {(df_pc['DAN+OTROS'] > 0).sum()} registros")
                print(f"   - Valores negativos: {(df_pc['DAN+OTROS'] < 0).sum()} registros")
                print(f"   - Valores cero: {(df_pc['DAN+OTROS'] == 0).sum()} registros")
                
            else:
                print("⚠️ No se encontraron las columnas CANTIDADASIGNADA o CANTIDADCARGADA para crear DAN+OTROS")
                columnas_cantidad = [col for col in df_pc.columns if 'CANTIDAD' in col.upper()]
                print(f"Columnas de cantidad disponibles: {columnas_cantidad}")
            
            # Mostrar estadísticas de clasificación
            total_calentadores = df_pc['Calentadores'].sum()
            total_escaleras = df_pc['Escaleras'].sum()
            total_otros = len(df_pc) - total_calentadores - total_escaleras
            
            print(f"📊 Estadísticas de clasificación:")
            print(f"   - Calentadores: {total_calentadores} registros")
            print(f"   - Escaleras: {total_escaleras} registros")
            print(f"   - Otros productos: {total_otros} registros")
            
            # Mostrar ejemplos de clasificación
            ejemplos_clasificacion = df_pc[[columna_producto, 'Calentadores', 'Escaleras']].dropna().head(5)
            if not ejemplos_clasificacion.empty:
                print(f"\n📝 Ejemplos de clasificación:")
                for _, row in ejemplos_clasificacion.iterrows():
                    tipo = "Calentador" if row['Calentadores'] == 1 else ("Escalera" if row['Escaleras'] == 1 else "Otro")
                    print(f"   Código: {row[columna_producto]} → {tipo}")
            
            # Mostrar algunos ejemplos de la transformación CONCA
            ejemplos = df_pc[[columna_lote, 'RUTA', columna_producto, 'CONCA', 'Calentadores', 'Escaleras', 'DAN+OTROS']].dropna().head(3)
            if not ejemplos.empty:
                print("📝 Ejemplos de transformación:")
                for _, row in ejemplos.iterrows():
                    tipo = "Calentador" if row['Calentadores'] == 1 else ("Escalera" if row['Escaleras'] == 1 else "Otro")
                    print(f"   {row[columna_lote]} → RUTA: {row['RUTA']} + TRIM(CLAVEPRODUCTO): {row[columna_producto]} = CONCA: {row['CONCA']} [{tipo}]")
        else:
            print("⚠️ No se encontraron las columnas LOTETERMINADO o TRIM(DE.CLAVEPRODUCTO) para crear CONCA")
            print(f"Columnas disponibles: {list(df_pc.columns)}")
            print("📝 Buscando columnas que contengan 'TRIM' y 'CLAVEPRODUCTO':")
            for col in df_pc.columns:
                if 'TRIM' in col.upper() and 'CLAVEPRODUCTO' in col.upper():
                    print(f"   - {col}")
        
        # PASO 4: RESUMEN DE RESULTADOS
        print(f"\n📊 RESULTADO CONSOLIDADO:")
        print("="*60)
        print(f"Total de registros: {df_pc.shape[0]}")
        print(f"Total de columnas: {df_pc.shape[1]}")
        
        print(f"\nColumnas en el DataFrame final:")
        for i, col in enumerate(df_pc.columns, 1):
            print(f"  {i}. {col}")
        
        # Mostrar resumen por origen
        print(f"\nResumen por archivo/hoja:")
        resumen_origen = df_pc.groupby(['ARCHIVO_ORIGEN', 'HOJA_ORIGEN']).size().reset_index(name='REGISTROS')
        print(resumen_origen.to_string(index=False))
        
        # Mostrar primeras filas
        print(f"\nPrimeras 5 filas del DataFrame consolidado:")
        print(df_pc.head().to_string())
        
        # PASO 5: EXPORTACIÓN A EXCEL
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        nombre_output = f'df_pc_consolidado_{timestamp}.xlsx'
        ruta_output = os.path.join(carpeta_outputs, nombre_output)
        
        try:
            # Crear DataFrame agrupado por CONCA para la segunda hoja
            if 'CONCA' in df_pc.columns and columna_asignada is not None and columna_cargada is not None:
                df_pc_agrupado = df_pc.groupby('CONCA').agg({
                    'RUTA': 'first',
                    columna_producto: 'first',
                    columna_asignada: 'sum',
                    columna_cargada: 'sum',
                    'DAN+OTROS': 'sum'
                }).reset_index()
                
                # Renombrar columnas para el reporte
                df_pc_agrupado = df_pc_agrupado.rename(columns={
                    columna_producto: 'TRIM(DE.CLAVEPRODUCTO)',
                    columna_asignada: 'Suma de CANTIDADASIGNADA',
                    columna_cargada: 'Suma de CANTIDADCARGADA',
                    'DAN+OTROS': 'Suma de DAN+OTROS'
                })
                
                print(f"\n✅ DataFrame agrupado creado:")
                print(f"   Registros agrupados: {df_pc_agrupado.shape[0]}")
                print(f"   Columnas: {list(df_pc_agrupado.columns)}")
            
            # Crear DataFrame agrupado por RUTA para la tercera hoja (Materiales)
            if 'RUTA' in df_pc.columns:
                df_materiales = df_pc.groupby('RUTA').agg({
                    'Escaleras': 'sum',
                    'Calentadores': 'sum'
                }).reset_index()
                
                print(f"\n✅ DataFrame de Materiales creado:")
                print(f"   Registros agrupados por RUTA: {df_materiales.shape[0]}")
                print(f"   Columnas: {list(df_materiales.columns)}")
                
                # Mostrar estadísticas de materiales por ruta
                print(f"📊 Estadísticas de Materiales por RUTA:")
                print(f"   - Total Escaleras: {df_materiales['Escaleras'].sum()} registros")
                print(f"   - Total Calentadores: {df_materiales['Calentadores'].sum()} registros")
                print(f"   - Rutas con Escaleras: {(df_materiales['Escaleras'] > 0).sum()} rutas")
                print(f"   - Rutas con Calentadores: {(df_materiales['Calentadores'] > 0).sum()} rutas")
            
            # DATOS PREPARADOS PARA EXPORTACIÓN FINAL
            print(f"\n✅ DATOS PC PREPARADOS:")
            if 'df_pc_agrupado' in locals() and 'df_materiales' in locals():
                print(f"📋 DataFrames creados:")
                print(f"   - df_pc_agrupado (Consolidado): {df_pc_agrupado.shape[0]} registros")
                print(f"   - df_materiales (Materiales): {df_materiales.shape[0]} registros")
            elif 'df_pc_agrupado' in locals():
                print(f"📋 DataFrame creado:")
                print(f"   - df_pc_agrupado (Consolidado): {df_pc_agrupado.shape[0]} registros")
            print(f"📄 Serán incluidos en el archivo Excel final consolidado")
            
        except Exception as e:
            print(f"❌ Error al exportar archivo: {e}")
    
    else:
        print("⚠️ No se pudieron procesar datos de ningún archivo.")
        
else:
    print("⚠️ No se encontraron archivos Excel ni CSV en la carpeta PC.")

print(f"\n🎉 PROCESAMIENTO PC COMPLETADO!")
print("="*60)

🔍 EXPLORANDO CARPETA PC...
Contenido de la carpeta: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Calculo de PC y DAN\Abastos\PC
--------------------------------------------------
1. exportcarga.xlsx (5437.5 KB)

Total de archivos encontrados: 1

Archivos Excel encontrados: 1
- exportcarga.xlsx


🔍 PROCESANDO ARCHIVOS PC (Excel y CSV)...

📂 Procesando: exportcarga.xlsx (Tipo: .xlsx)
   Hojas encontradas: 2 -> ['Exportar Hoja de Trabajo', 'SQL']


c:\Users\igcamposg\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


   ✅ Hoja 'Exportar Hoja de Trabajo': 89909 filas procesadas
   ✅ Hoja 'SQL': 1 filas procesadas
🔍 Columnas identificadas:
   - Lote: LOTETERMINADO
   - Producto: TRIM(DE.CLAVEPRODUCTO)


c:\Users\igcamposg\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")



✅ Columnas RUTA y CONCA creadas exitosamente
   - CONCA estandarizada como string sin espacios
✅ Columnas de clasificación creadas: Calentadores y Escaleras
🔍 Columnas de cantidad identificadas:
   - Cantidad Asignada: CANTIDADASIGNADA
   - Cantidad Cargada: CANTIDADCARGADA
✅ Columna DAN+OTROS creada (CANTIDADASIGNADA - CANTIDADCARGADA)
📊 Estadísticas de DAN+OTROS:
   - Total DAN+OTROS: 160,028
   - Promedio: 1.78
   - Valores positivos: 279 registros
   - Valores negativos: 0 registros
   - Valores cero: 89630 registros
📊 Estadísticas de clasificación:
   - Calentadores: 36 registros
   - Escaleras: 405 registros
   - Otros productos: 89469 registros

📝 Ejemplos de clasificación:
   Código: 103143 → Otro
   Código: 11004 → Otro
   Código: 12520 → Otro
   Código: 12556 → Otro
   Código: 12587 → Otro
📝 Ejemplos de transformación:
   E260503072028             → RUTA: 72028 + TRIM(CLAVEPRODUCTO): 103143 = CONCA: 72028103143 [Otro]
   E260503072028             → RUTA: 72028 + TRIM(CLAVEPR

03. Calculo DAN Total

In [3]:
# ==================== CALCULO DAN TOTAL - MERGE DE PC Y DAN ====================
print("🔗 CREANDO DF_DAN_TOTAL CON MERGE PC + DAN...")
print("="*60)

# Verificar que existan los DataFrames necesarios
if 'df_pc_agrupado' in locals() and 'df_danes_agrupado' in locals():
    print("✅ DataFrames encontrados:")
    print(f"   - df_pc_agrupado: {df_pc_agrupado.shape[0]} registros")
    print(f"   - df_danes_agrupado: {df_danes_agrupado.shape[0]} registros")
    
    # Crear df_dan_total seleccionando las columnas especificadas de df_pc_agrupado
    columnas_seleccionadas = ['CONCA', 'RUTA', 'TRIM(DE.CLAVEPRODUCTO)', 'Suma de CANTIDADCARGADA', 'Suma de DAN+OTROS']
    df_dan_total = df_pc_agrupado[columnas_seleccionadas].copy()
    
    print(f"\n📋 DataFrame base creado:")
    print(f"   - Registros: {df_dan_total.shape[0]}")
    print(f"   - Columnas: {list(df_dan_total.columns)}")
    
    # DIAGNÓSTICO Y CORRECCIÓN DE TIPOS DE DATOS PARA MERGE
    print(f"\n🔍 DIAGNÓSTICO DE TIPOS DE DATOS PARA MERGE:")
    print(f"   - df_pc_agrupado['CONCA'] tipo: {df_pc_agrupado['CONCA'].dtype}")
    print(f"   - df_danes_agrupado['CONCA'] tipo: {df_danes_agrupado['CONCA'].dtype}")
    
    # Verificar contenido de espacios en blanco
    pc_with_spaces = df_pc_agrupado['CONCA'].astype(str).str.contains(' ', na=False).sum()
    danes_with_spaces = df_danes_agrupado['CONCA'].astype(str).str.contains(' ', na=False).sum()
    print(f"   - PC registros con espacios: {pc_with_spaces}")
    print(f"   - DANES registros con espacios: {danes_with_spaces}")
    
    # Verificar coincidencias antes de merge
    coincidencias_antes = set(df_pc_agrupado['CONCA'].astype(str).str.strip()) & set(df_danes_agrupado['CONCA'].astype(str).str.strip())
    print(f"   - Coincidencias potenciales: {len(coincidencias_antes)}")
    
    # ESTANDARIZAR TIPOS DE DATOS Y LIMPIAR ESPACIOS
    print(f"\n🔧 ESTANDARIZANDO TIPOS DE DATOS:")
    
    # Convertir ambas columnas CONCA a string y limpiar espacios
    df_dan_total['CONCA'] = df_dan_total['CONCA'].astype(str).str.strip()
    df_danes_agrupado['CONCA'] = df_danes_agrupado['CONCA'].astype(str).str.strip()
    
    print(f"   ✅ Ambas columnas CONCA convertidas a string sin espacios")
    print(f"   - df_dan_total['CONCA'] tipo: {df_dan_total['CONCA'].dtype}")
    print(f"   - df_danes_agrupado['CONCA'] tipo: {df_danes_agrupado['CONCA'].dtype}")
    
    # Realizar el merge tipo VLOOKUP
    # Hacer merge con df_danes_agrupado para traer la columna CANTIDAD
    df_dan_total = df_dan_total.merge(
        df_danes_agrupado[['CONCA', 'CANTIDAD']], 
        on='CONCA', 
        how='left'
    )
    
    # Renombrar la columna CANTIDAD a "Reporte DAN"
    df_dan_total = df_dan_total.rename(columns={'CANTIDAD': 'Reporte DAN'})
    
    # Crear columna "Tipo" con la lógica especificada
    def calcular_tipo(row):
        suma_dan_otros = row['Suma de DAN+OTROS'] if pd.notna(row['Suma de DAN+OTROS']) else 0
        reporte_dan = row['Reporte DAN'] if pd.notna(row['Reporte DAN']) else 0
        
        # Lógica corregida: SI(Y([Suma de DAN+OTROS]=[Reporte DAN],[Reporte DAN]=0),"Sin DAN",
        if suma_dan_otros == reporte_dan and reporte_dan == 0:
            return "Sin DAN"
        # SI(Y([Suma de DAN+OTROS]=[Reporte DAN],[Reporte DAN]>0),"DAN Planeado",
        elif suma_dan_otros == reporte_dan and reporte_dan > 0:
            return "DAN Planeado"
        # SI(Y([Reporte DAN]>[Suma de DAN+OTROS],[Suma de DAN+OTROS]=0),"DAN No Planeado",
        elif reporte_dan > suma_dan_otros and suma_dan_otros == 0:
            return "DAN No Planeado"
        # SI([Reporte DAN]=0,"DAN No Reportado","DAN Mixto"))))
        elif reporte_dan == 0:
            return "DAN No Reportado"
        else:
            return "DAN Mixto"
    
    # Aplicar la función a cada fila
    df_dan_total['Tipo'] = df_dan_total.apply(calcular_tipo, axis=1)
    
    print(f"\n✅ Columna 'Tipo' creada exitosamente")
    
    # Crear las 3 columnas calculadas adicionales
    
    # 1. DAN Planeado: =SI(O([TIPO]="DAN Planeado",[Tipo]="DAN Mixto"),[Suma de DAN+OTROS],"-")
    def calcular_dan_planeado(row):
        if row['Tipo'] in ['DAN Planeado', 'DAN Mixto']:
            return row['Suma de DAN+OTROS'] if pd.notna(row['Suma de DAN+OTROS']) else 0
        else:
            return "-"
    
    # 2. DAN No Planeado: =SI([Tipo]="DAN No Planeado",[Reporte DAN],SI([Tipo]="DAN Mixto",SI([Reporte DAN]>[Suma de DAN+OTROS],[Reporte DAN]-[Suma de DAN+OTROS],0),"-"))
    def calcular_dan_no_planeado(row):
        reporte_dan = row['Reporte DAN'] if pd.notna(row['Reporte DAN']) else 0
        suma_dan_otros = row['Suma de DAN+OTROS'] if pd.notna(row['Suma de DAN+OTROS']) else 0
        
        if row['Tipo'] == 'DAN No Planeado':
            return reporte_dan
        elif row['Tipo'] == 'DAN Mixto':
            if reporte_dan > suma_dan_otros:
                return reporte_dan - suma_dan_otros
            else:
                return 0
        else:
            return "-"
    
    # 3. DAN No Reportado: =SI([Tipo]="DAN No Reportado",[Suma de DAN+OTROS],SI([Suma de DAN+OTROS]>[Reporte DAN],[Suma de DAN+OTROS]-[Reporte DAN],"-"))
    def calcular_dan_no_reportado(row):
        reporte_dan = row['Reporte DAN'] if pd.notna(row['Reporte DAN']) else 0
        suma_dan_otros = row['Suma de DAN+OTROS'] if pd.notna(row['Suma de DAN+OTROS']) else 0
        
        if row['Tipo'] == 'DAN No Reportado':
            return suma_dan_otros
        elif suma_dan_otros > reporte_dan:
            return suma_dan_otros - reporte_dan
        else:
            return "-"
    
    # Aplicar las funciones
    df_dan_total['DAN Planeado'] = df_dan_total.apply(calcular_dan_planeado, axis=1)
    df_dan_total['DAN No Planeado'] = df_dan_total.apply(calcular_dan_no_planeado, axis=1)
    df_dan_total['DAN No Reportado'] = df_dan_total.apply(calcular_dan_no_reportado, axis=1)
    
    print(f"✅ Columnas calculadas creadas: 'DAN Planeado', 'DAN No Planeado', 'DAN No Reportado'")
    
    print(f"\n✅ Merge completado:")
    print(f"   - Registros finales: {df_dan_total.shape[0]}")
    print(f"   - Columnas finales: {list(df_dan_total.columns)}")
    
    # Mostrar estadísticas del merge
    registros_con_dan = df_dan_total['Reporte DAN'].notna().sum()
    registros_sin_dan = df_dan_total['Reporte DAN'].isna().sum()
    
    print(f"\n📊 Estadísticas del merge:")
    print(f"   - Registros con DAN encontrado: {registros_con_dan}")
    print(f"   - Registros sin DAN (NaN): {registros_sin_dan}")
    print(f"   - Porcentaje de coincidencias: {(registros_con_dan/len(df_dan_total)*100):.1f}%")
    
    # Mostrar estadísticas de la columna Tipo
    tipo_counts = df_dan_total['Tipo'].value_counts()
    print(f"\n📊 Distribución por Tipo:")
    for tipo, count in tipo_counts.items():
        porcentaje = (count/len(df_dan_total)*100)
        print(f"   - {tipo}: {count} registros ({porcentaje:.1f}%)")
    
    # Mostrar estadísticas de las columnas calculadas
    print(f"\n📊 Estadísticas de columnas calculadas:")
    
    # DAN Planeado
    dan_planeado_numeric = pd.to_numeric(df_dan_total['DAN Planeado'], errors='coerce')
    dan_planeado_count = dan_planeado_numeric.notna().sum()
    dan_planeado_total = dan_planeado_numeric.sum()
    print(f"   - DAN Planeado: {dan_planeado_count} registros con valores, Total: {dan_planeado_total:,.0f}")
    
    # DAN No Planeado
    dan_no_planeado_numeric = pd.to_numeric(df_dan_total['DAN No Planeado'], errors='coerce')
    dan_no_planeado_count = dan_no_planeado_numeric.notna().sum()
    dan_no_planeado_total = dan_no_planeado_numeric.sum()
    print(f"   - DAN No Planeado: {dan_no_planeado_count} registros con valores, Total: {dan_no_planeado_total:,.0f}")
    
    # DAN No Reportado
    dan_no_reportado_numeric = pd.to_numeric(df_dan_total['DAN No Reportado'], errors='coerce')
    dan_no_reportado_count = dan_no_reportado_numeric.notna().sum()
    dan_no_reportado_total = dan_no_reportado_numeric.sum()
    print(f"   - DAN No Reportado: {dan_no_reportado_count} registros con valores, Total: {dan_no_reportado_total:,.0f}")
    
    if registros_con_dan > 0:
        total_dan = df_dan_total['Reporte DAN'].sum()
        promedio_dan = df_dan_total['Reporte DAN'].mean()
        print(f"\n📈 Estadísticas numéricas:")
        print(f"   - Total Reporte DAN: {total_dan:,.0f}")
        print(f"   - Promedio Reporte DAN: {promedio_dan:.2f}")
        print(f"   - Total DAN+OTROS: {df_dan_total['Suma de DAN+OTROS'].sum():,.0f}")
        print(f"   - Total Cantidad Cargada: {df_dan_total['Suma de CANTIDADCARGADA'].sum():,.0f}")
    
    # Mostrar primeros registros del resultado
    print(f"\n📝 Primeros 10 registros de df_dan_total:")
    print(df_dan_total.head(10).to_string(index=False))
    
    # Mostrar algunos ejemplos de coincidencias encontradas
    ejemplos_coincidencias = df_dan_total[df_dan_total['Reporte DAN'].notna()].head(5)
    if not ejemplos_coincidencias.empty:
        print(f"\n🎯 Ejemplos de coincidencias encontradas:")
        for _, row in ejemplos_coincidencias.iterrows():
            print(f"   CONCA: {row['CONCA']} → RUTA: {row['RUTA']} → DAN: {row['Reporte DAN']:,.0f} → Tipo: {row['Tipo']}")
    
    # Mostrar algunos ejemplos de registros sin coincidencia
    ejemplos_sin_coincidencia = df_dan_total[df_dan_total['Reporte DAN'].isna()].head(3)
    if not ejemplos_sin_coincidencia.empty:
        print(f"\n⚠️ Ejemplos de registros sin coincidencia en DAN:")
        for _, row in ejemplos_sin_coincidencia.iterrows():
            print(f"   CONCA: {row['CONCA']} → RUTA: {row['RUTA']} → Tipo: {row['Tipo']}")
    
    # Mostrar ejemplos por cada tipo de DAN
    print(f"\n📝 Ejemplos por tipo de DAN con columnas calculadas:")
    for tipo in df_dan_total['Tipo'].unique():
        ejemplo = df_dan_total[df_dan_total['Tipo'] == tipo].head(1)
        if not ejemplo.empty:
            row = ejemplo.iloc[0]
            cantidad_cargada = row['Suma de CANTIDADCARGADA'] if pd.notna(row['Suma de CANTIDADCARGADA']) else 0
            reporte_dan = row['Reporte DAN'] if pd.notna(row['Reporte DAN']) else 0
            suma_dan_otros = row['Suma de DAN+OTROS'] if pd.notna(row['Suma de DAN+OTROS']) else 0
            print(f"   {tipo}:")
            print(f"     CONCA={row['CONCA']}, Cargada={cantidad_cargada:,.0f}, DAN={reporte_dan:,.0f}, DAN+OTROS={suma_dan_otros:,.0f}")
            print(f"     → DAN Planeado: {row['DAN Planeado']}, DAN No Planeado: {row['DAN No Planeado']}, DAN No Reportado: {row['DAN No Reportado']}")
    
    # Mostrar resumen de todas las columnas calculadas por tipo
    print(f"\n📋 Resumen de columnas calculadas por tipo:")
    resumen_por_tipo = df_dan_total.groupby('Tipo').agg({
        'DAN Planeado': lambda x: pd.to_numeric(x, errors='coerce').sum(),
        'DAN No Planeado': lambda x: pd.to_numeric(x, errors='coerce').sum(),
        'DAN No Reportado': lambda x: pd.to_numeric(x, errors='coerce').sum()
    }).round(0)
    print(resumen_por_tipo.to_string())
    
    # CREAR DF_DANES_TOTAL AGRUPADO POR RUTA Y TRIM(DE.CLAVEPRODUCTO)
    print(f"\n📊 CREANDO DF_DANES_TOTAL AGRUPADO POR RUTA Y TRIM(DE.CLAVEPRODUCTO)...")
    print("-" * 50)
    
    # Agrupar por RUTA y TRIM(DE.CLAVEPRODUCTO) y sumar las columnas calculadas
    df_danes_total = df_dan_total.groupby(['RUTA', 'TRIM(DE.CLAVEPRODUCTO)']).agg({
        'DAN Planeado': lambda x: pd.to_numeric(x, errors='coerce').sum(),
        'DAN No Planeado': lambda x: pd.to_numeric(x, errors='coerce').sum(),
        'DAN No Reportado': lambda x: pd.to_numeric(x, errors='coerce').sum()
    }).reset_index()
    
    # Renombrar las columnas sumadas
    df_danes_total = df_danes_total.rename(columns={
        'DAN Planeado': 'Suma de DAN Planeado',
        'DAN No Planeado': 'Suma de DAN No Planeado',
        'DAN No Reportado': 'Suma de DAN No Reportado'
    })
    
    print(f"✅ DataFrame df_danes_total creado:")
    print(f"   - Registros agrupados por RUTA + TRIM(DE.CLAVEPRODUCTO): {df_danes_total.shape[0]}")
    print(f"   - Columnas: {list(df_danes_total.columns)}")
    
    # Mostrar estadísticas del agrupamiento
    print(f"\n📈 Estadísticas del agrupamiento:")
    print(f"   - Combinaciones únicas RUTA + PRODUCTO: {df_danes_total.shape[0]}")
    print(f"   - Total Suma de DAN Planeado: {df_danes_total['Suma de DAN Planeado'].sum():,.0f}")
    print(f"   - Total Suma de DAN No Planeado: {df_danes_total['Suma de DAN No Planeado'].sum():,.0f}")
    print(f"   - Total Suma de DAN No Reportado: {df_danes_total['Suma de DAN No Reportado'].sum():,.0f}")
    
    # Mostrar estadísticas adicionales
    rutas_unicas = df_danes_total['RUTA'].nunique()
    productos_unicos = df_danes_total['TRIM(DE.CLAVEPRODUCTO)'].nunique()
    print(f"   - Rutas únicas: {rutas_unicas}")
    print(f"   - Productos únicos: {productos_unicos}")
    
    # Mostrar primeros registros del resultado agrupado
    print(f"\n📝 Primeros 10 registros de df_danes_total:")
    print(df_danes_total.head(10).to_string(index=False))
    
    # TRANSFORMACIÓN: INVERTIR COLUMNAS A FORMATO LARGO (UNPIVOT)
    print(f"\n🔄 TRANSFORMANDO DF_DANES_TOTAL A FORMATO LARGO...")
    print("-" * 50)
    
    # Crear una lista para almacenar los datos transformados
    datos_transformados = []
    
    # Iterar sobre cada fila del DataFrame agrupado
    for _, row in df_danes_total.iterrows():
        ruta = row['RUTA']
        producto = row['TRIM(DE.CLAVEPRODUCTO)']
        
        # Crear una fila para cada tipo de DAN si tiene valor > 0
        if row['Suma de DAN Planeado'] > 0:
            datos_transformados.append({
                'DAN': 'Planeado',
                'RUTA': ruta,
                'TRIM(DE.CLAVEPRODUCTO)': producto,
                'Cantidad a enviar': row['Suma de DAN Planeado']
            })
        
        if row['Suma de DAN No Planeado'] > 0:
            datos_transformados.append({
                'DAN': 'No Planeado',
                'RUTA': ruta,
                'TRIM(DE.CLAVEPRODUCTO)': producto,
                'Cantidad a enviar': row['Suma de DAN No Planeado']
            })
        
        if row['Suma de DAN No Reportado'] > 0:
            datos_transformados.append({
                'DAN': 'No Surtido',
                'RUTA': ruta,
                'TRIM(DE.CLAVEPRODUCTO)': producto,
                'Cantidad a enviar': row['Suma de DAN No Reportado']
            })
    
    # Crear el nuevo DataFrame transformado
    df_danes_total_transformado = pd.DataFrame(datos_transformados)
    
    # Reordenar las columnas según especificación: DAN, RUTA, TRIM(DE.CLAVEPRODUCTO), Cantidad a enviar
    columnas_finales = ['DAN', 'RUTA', 'TRIM(DE.CLAVEPRODUCTO)', 'Cantidad a enviar']
    df_danes_total_final = df_danes_total_transformado[columnas_finales].copy()
    
    print(f"✅ Transformación completada:")
    print(f"   - Registros originales (formato ancho): {df_danes_total.shape[0]}")
    print(f"   - Registros transformados (formato largo): {df_danes_total_final.shape[0]}")
    print(f"   - Columnas finales: {list(df_danes_total_final.columns)}")
    
    # Mostrar estadísticas de la transformación
    print(f"\n📊 Distribución por tipo de DAN:")
    dan_counts = df_danes_total_final['DAN'].value_counts()
    for tipo, count in dan_counts.items():
        total_cantidad = df_danes_total_final[df_danes_total_final['DAN'] == tipo]['Cantidad a enviar'].sum()
        print(f"   - {tipo}: {count} registros, Total cantidad: {total_cantidad:,.0f}")
    
    # Mostrar primeros registros transformados
    print(f"\n📝 Primeros 10 registros de df_danes_total_final:")
    print(df_danes_total_final.head(10).to_string(index=False))
    
    # Actualizar df_danes_total para usar en la exportación
    df_danes_total = df_danes_total_final.copy()
    
    # PASO ADICIONAL: CONVERSIÓN A FORMATO NUMÉRICO PARA BÚSQUEDAS EN DICCIONARIOS
    print(f"\n🔢 CONVIRTIENDO RUTA Y TRIM(DE.CLAVEPRODUCTO) A FORMATO NUMÉRICO...")
    print("-" * 60)
    
    # Convertir RUTA a numérico
    print(f"📊 Convirtiendo columna RUTA:")
    print(f"   - Tipo original: {df_danes_total['RUTA'].dtype}")
    print(f"   - Valores únicos originales (primeros 5): {df_danes_total['RUTA'].unique()[:5]}")
    
    # Función para convertir RUTA a numérico de forma segura
    def convertir_ruta_a_numero(valor):
        try:
            # Remover espacios y convertir a string primero
            valor_str = str(valor).strip()
            # Intentar convertir a entero
            return int(float(valor_str))
        except (ValueError, TypeError):
            print(f"   ⚠️ No se pudo convertir RUTA: '{valor}' a número")
            return valor  # Mantener valor original si no se puede convertir
    
    df_danes_total['RUTA'] = df_danes_total['RUTA'].apply(convertir_ruta_a_numero)
    print(f"   - Tipo después de conversión: {df_danes_total['RUTA'].dtype}")
    print(f"   - Valores únicos convertidos (primeros 5): {df_danes_total['RUTA'].unique()[:5]}")
    
    # Convertir TRIM(DE.CLAVEPRODUCTO) a numérico
    print(f"\n📊 Convirtiendo columna TRIM(DE.CLAVEPRODUCTO):")
    print(f"   - Tipo original: {df_danes_total['TRIM(DE.CLAVEPRODUCTO)'].dtype}")
    print(f"   - Valores únicos originales (primeros 5): {df_danes_total['TRIM(DE.CLAVEPRODUCTO)'].unique()[:5]}")
    
    # Función para convertir TRIM(DE.CLAVEPRODUCTO) a numérico de forma segura
    def convertir_producto_a_numero(valor):
        try:
            # Remover espacios y convertir a string primero
            valor_str = str(valor).strip()
            # Intentar convertir a entero
            return int(float(valor_str))
        except (ValueError, TypeError):
            print(f"   ⚠️ No se pudo convertir PRODUCTO: '{valor}' a número")
            return valor  # Mantener valor original si no se puede convertir
    
    df_danes_total['TRIM(DE.CLAVEPRODUCTO)'] = df_danes_total['TRIM(DE.CLAVEPRODUCTO)'].apply(convertir_producto_a_numero)
    print(f"   - Tipo después de conversión: {df_danes_total['TRIM(DE.CLAVEPRODUCTO)'].dtype}")
    print(f"   - Valores únicos convertidos (primeros 5): {df_danes_total['TRIM(DE.CLAVEPRODUCTO)'].unique()[:5]}")
    
    # Verificar que las conversiones fueron exitosas
    rutas_numericas = pd.to_numeric(df_danes_total['RUTA'], errors='coerce').notna().sum()
    productos_numericos = pd.to_numeric(df_danes_total['TRIM(DE.CLAVEPRODUCTO)'], errors='coerce').notna().sum()
    total_registros = len(df_danes_total)
    
    print(f"\n✅ RESULTADOS DE CONVERSIÓN:")
    print(f"   - RUTA: {rutas_numericas}/{total_registros} registros convertidos exitosamente ({(rutas_numericas/total_registros*100):.1f}%)")
    print(f"   - TRIM(DE.CLAVEPRODUCTO): {productos_numericos}/{total_registros} registros convertidos exitosamente ({(productos_numericos/total_registros*100):.1f}%)")
    
    if rutas_numericas == total_registros and productos_numericos == total_registros:
        print(f"   🎉 ¡Todas las conversiones fueron exitosas!")
    else:
        print(f"   ⚠️ Algunos valores no se pudieron convertir - revisar datos originales")
        
        # Mostrar ejemplos de valores que no se pudieron convertir
        rutas_problematicas = df_danes_total[pd.to_numeric(df_danes_total['RUTA'], errors='coerce').isna()]
        if not rutas_problematicas.empty:
            print(f"   📝 Ejemplos de RUTA problemáticas:")
            for valor in rutas_problematicas['RUTA'].unique()[:3]:
                print(f"     - '{valor}' (tipo: {type(valor)})")
        
        productos_problematicos = df_danes_total[pd.to_numeric(df_danes_total['TRIM(DE.CLAVEPRODUCTO)'], errors='coerce').isna()]
        if not productos_problematicos.empty:
            print(f"   📝 Ejemplos de PRODUCTOS problemáticos:")
            for valor in productos_problematicos['TRIM(DE.CLAVEPRODUCTO)'].unique()[:3]:
                print(f"     - '{valor}' (tipo: {type(valor)})")
    
    # Mostrar algunos ejemplos de los datos finales listos para búsquedas
    print(f"\n📝 EJEMPLOS DE DATOS PREPARADOS PARA BÚSQUEDAS:")
    ejemplos_finales = df_danes_total[['DAN', 'RUTA', 'TRIM(DE.CLAVEPRODUCTO)', 'Cantidad a enviar']].head(3)
    for _, row in ejemplos_finales.iterrows():
        print(f"   {row['DAN']} → RUTA: {row['RUTA']} (tipo: {type(row['RUTA'])}) + PRODUCTO: {row['TRIM(DE.CLAVEPRODUCTO)']} (tipo: {type(row['TRIM(DE.CLAVEPRODUCTO)'])}) → Cantidad: {row['Cantidad a enviar']}")
    
    print(f"\n✅ DATOS PREPARADOS PARA EXPORTACIÓN FINAL:")
    print(f"📋 df_danes_total_final: {df_danes_total.shape[0]} registros")
    print(f"📄 Ejecuta la celda 'Exportación Excel' al final para generar el archivo consolidado")
    
else:
    print("❌ Error: No se encontraron los DataFrames necesarios")
    if 'df_pc_agrupado' not in locals():
        print("   - Falta: df_pc_agrupado")
    if 'df_danes_agrupado' not in locals():
        print("   - Falta: df_danes_agrupado")
    print("   Ejecuta primero las celdas anteriores para generar los DataFrames.")

print(f"\n🎉 CALCULO DAN TOTAL COMPLETADO!")
print("="*60)

🔗 CREANDO DF_DAN_TOTAL CON MERGE PC + DAN...
✅ DataFrames encontrados:
   - df_pc_agrupado: 85728 registros
   - df_danes_agrupado: 259 registros

📋 DataFrame base creado:
   - Registros: 85728
   - Columnas: ['CONCA', 'RUTA', 'TRIM(DE.CLAVEPRODUCTO)', 'Suma de CANTIDADCARGADA', 'Suma de DAN+OTROS']

🔍 DIAGNÓSTICO DE TIPOS DE DATOS PARA MERGE:
   - df_pc_agrupado['CONCA'] tipo: object
   - df_danes_agrupado['CONCA'] tipo: object
   - PC registros con espacios: 0
   - DANES registros con espacios: 0
   - Coincidencias potenciales: 249

🔧 ESTANDARIZANDO TIPOS DE DATOS:
   ✅ Ambas columnas CONCA convertidas a string sin espacios
   - df_dan_total['CONCA'] tipo: object
   - df_danes_agrupado['CONCA'] tipo: object

✅ Columna 'Tipo' creada exitosamente
✅ Columnas calculadas creadas: 'DAN Planeado', 'DAN No Planeado', 'DAN No Reportado'

✅ Merge completado:
   - Registros finales: 85728
   - Columnas finales: ['CONCA', 'RUTA', 'TRIM(DE.CLAVEPRODUCTO)', 'Suma de CANTIDADCARGADA', 'Suma de DAN+

04. Simulador de DANES

In [4]:
# ==================== SIMULADOR DE EMPAQUES Y ESPECIFICACIONES ==================
import pandas as pd
import pprint
import os
from datetime import datetime

print("🔧 INICIANDO SIMULADOR DE EMPAQUES...")
print("="*60)

# --- 1. CONFIGURACIÓN ---
archivo_especificaciones = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Calculo de PC y DAN\ZQDIS.xlsx'
hoja_especificaciones = 'Especificaciones'

# --- 2. LECTURA Y CREACIÓN DEL DICCIONARIO MAESTRO ---
try:
    print("📖 Leyendo archivo de especificaciones...")
    df_especificaciones = pd.read_excel(archivo_especificaciones, sheet_name=hoja_especificaciones)
    print(f"✅ Archivo leído correctamente: {len(df_especificaciones)} registros encontrados")

    # Mapeo de columnas para estandarizar nombres
    columnas_a_renombrar = {
        'Piezas por presentación': 'Piezas',
        'Volumen (cm3)': 'Volumen',
        'Peso neto (kg)': 'Peso'
    }
    df_especificaciones = df_especificaciones.rename(columns=columnas_a_renombrar)

    # Seleccionar solo columnas necesarias y crear diccionario optimizado
    columnas_necesarias = ['SKU', 'Presentación', 'Piezas', 'Volumen', 'Peso']
    df_final = df_especificaciones[columnas_necesarias]
    
    # Eliminar duplicados: Si hay duplicados, se queda con el primero que encuentre
    df_final = df_final.drop_duplicates(subset=['SKU', 'Presentación'], keep='first')
    
    specs_map = df_final.set_index(['SKU', 'Presentación']).to_dict('index')
    
    print(f"✅ Diccionario de especificaciones creado: {len(specs_map)} entradas únicas")
    
    # Mostrar ejemplos del diccionario
    print("\n📝 Ejemplos del diccionario de especificaciones:")
    primeros_elementos = list(specs_map.items())[:3]
    for elemento in primeros_elementos:
        print(f"   {elemento[0]} → {elemento[1]}")

except FileNotFoundError:
    print(f"❌ ERROR: No se encontró el archivo '{archivo_especificaciones}'")
    exit()
except Exception as e:
    print(f"❌ ERROR al crear diccionario: {e}")
    exit()

# --- 3. FUNCIÓN DE OPTIMIZACIÓN DE EMPAQUES ---
def calcular_empaque_completo(fila, specs_map):
    """
    Calcula la distribución óptima de empaques para una cantidad específica
    usando estrategia greedy: Pallets → Masters → Inners → Piezas
    """
    codigo = fila['TRIM(DE.CLAVEPRODUCTO)']
    cantidad_a_enviar = fila['Cantidad a enviar']
    
    # Obtener especificaciones por tipo de empaque
    pallet_info = specs_map.get((codigo, 'Pallet'), {})
    master_info = specs_map.get((codigo, 'Master'), {})
    inner_info = specs_map.get((codigo, 'Inner'), {})
    pieza_info = specs_map.get((codigo, 'Pieza'), {})

    # Extraer datos de cada tipo de empaque
    piezas_pallet = pallet_info.get('Piezas', 0)
    volumen_pallet = pallet_info.get('Volumen', 0)
    peso_pallet = pallet_info.get('Peso', 0)
    
    piezas_master = master_info.get('Piezas', 0)
    volumen_master = master_info.get('Volumen', 0)
    peso_master = master_info.get('Peso', 0)

    piezas_inner = inner_info.get('Piezas', 0)
    volumen_inner = inner_info.get('Volumen', 0)
    peso_inner = inner_info.get('Peso', 0)

    piezas_unitario = pieza_info.get('Piezas', 1)  # Default 1 para piezas
    volumen_unitario = pieza_info.get('Volumen', 0)
    peso_unitario = pieza_info.get('Peso', 0)

    # Algoritmo greedy de optimización de empaques
    cantidad_restante = cantidad_a_enviar
    
    # 1. Pallets
    cant_pallets = cantidad_restante // piezas_pallet if piezas_pallet > 0 else 0
    resto_pallet = cantidad_restante % piezas_pallet if piezas_pallet > 0 else cantidad_restante
    
    # 2. Masters
    cant_masters = resto_pallet // piezas_master if piezas_master > 0 else 0
    resto_master = resto_pallet % piezas_master if piezas_master > 0 else resto_pallet
    
    # 3. Inners
    cant_inners = resto_master // piezas_inner if piezas_inner > 0 else 0
    resto_inner = resto_master % piezas_inner if piezas_inner > 0 else resto_master
    
    # 4. Piezas sueltas
    cant_piezas = resto_inner // piezas_unitario if piezas_unitario > 0 else 0
    resto_final = resto_inner % piezas_unitario if piezas_unitario > 0 else resto_inner

    # Cálculos de volumen y peso totales
    volumen_total = (cant_pallets * volumen_pallet) + (cant_masters * volumen_master) + \
                   (cant_inners * volumen_inner) + (cant_piezas * volumen_unitario)
    peso_total = (cant_pallets * peso_pallet) + (cant_masters * peso_master) + \
                (cant_inners * peso_inner) + (cant_piezas * peso_unitario)

    # Check de validación
    check_warning = "⚠️ Aguas" if resto_final != 0 else "✅ OK"

    return pd.Series([
        volumen_total, peso_total, cant_pallets, cant_masters, cant_inners, cant_piezas,
        piezas_pallet, volumen_pallet, peso_pallet, resto_pallet, piezas_master,
        volumen_master, peso_master, resto_master, piezas_inner, volumen_inner,
        peso_inner, resto_inner, piezas_unitario, volumen_unitario, peso_unitario,
        resto_final, check_warning
    ])

# --- 4. DEMOSTRACIÓN CON DATOS DE EJEMPLO ---
print(f"\n🧪 PRUEBA RÁPIDA DEL SIMULADOR:")
print("-" * 40)

# Buscar un SKU real del diccionario para demostrar
skus_disponibles = [key[0] for key in specs_map.keys()]
if skus_disponibles:
    sku_ejemplo = skus_disponibles[0]
    print(f"📦 Usando SKU de ejemplo: {sku_ejemplo}")
    
    # Crear datos de prueba
    datos_prueba = pd.DataFrame({
        'TRIM(DE.CLAVEPRODUCTO)': [sku_ejemplo],
        'Cantidad a enviar': [1250]  # Cantidad de ejemplo
    })
    
    # Ejecutar cálculo
    resultado_prueba = calcular_empaque_completo(datos_prueba.iloc[0], specs_map)
    
    print(f"🎯 Resultado para {1250} piezas:")
    print(f"   Pallets: {resultado_prueba[2]}")
    print(f"   Masters: {resultado_prueba[3]}")
    print(f"   Inners: {resultado_prueba[4]}")
    print(f"   Piezas: {resultado_prueba[5]}")
    print(f"   Volumen total: {resultado_prueba[0]:.2f} m³")
    print(f"   Peso total: {resultado_prueba[1]:.2f} kg")
    print(f"   Estado: {resultado_prueba[22]}")

# --- 5. PROCESAMIENTO AUTOMÁTICO CON DATOS DE DAN ---
print(f"\n🚀 PROCESANDO DATOS DE DAN CON SIMULADOR DE EMPAQUES...")
print("-" * 60)

# Verificar si tenemos los datos de DAN transformados disponibles
if 'df_danes_total_final' in locals() or 'df_danes_total' in locals():
    # Usar el DataFrame de DAN disponible
    if 'df_danes_total_final' in locals():
        df_pedidos = df_danes_total_final.copy()
        print(f"✅ Usando df_danes_total_final: {df_pedidos.shape[0]} registros")
    else:
        df_pedidos = df_danes_total.copy()
        print(f"✅ Usando df_danes_total: {df_pedidos.shape[0]} registros")
    
    # Limpiar la columna de código de producto
    df_pedidos['TRIM(DE.CLAVEPRODUCTO)'] = df_pedidos['TRIM(DE.CLAVEPRODUCTO)'].astype(str).str.strip()
    
    # CONVERSIÓN A FORMATO NUMÉRICO PARA BÚSQUEDAS EN DICCIONARIOS
    print(f"\n🔢 CONVIRTIENDO DATOS A FORMATO NUMÉRICO PARA BÚSQUEDAS...")
    print("-" * 50)
    
    # Mostrar tipos originales
    print(f"📊 Tipos de datos originales:")
    print(f"   - RUTA: {df_pedidos['RUTA'].dtype}")
    print(f"   - TRIM(DE.CLAVEPRODUCTO): {df_pedidos['TRIM(DE.CLAVEPRODUCTO)'].dtype}")
    
    # Convertir RUTA a numérico
    def convertir_a_numero_seguro(valor):
        try:
            # Remover espacios y convertir a string primero
            valor_str = str(valor).strip()
            # Intentar convertir a entero
            return int(float(valor_str))
        except (ValueError, TypeError):
            return valor  # Mantener valor original si no se puede convertir
    
    # Aplicar conversión a ambas columnas
    df_pedidos['RUTA'] = df_pedidos['RUTA'].apply(convertir_a_numero_seguro)
    df_pedidos['TRIM(DE.CLAVEPRODUCTO)'] = df_pedidos['TRIM(DE.CLAVEPRODUCTO)'].apply(convertir_a_numero_seguro)
    
    # Verificar tipos después de conversión
    print(f"\n📊 Tipos de datos después de conversión:")
    print(f"   - RUTA: {df_pedidos['RUTA'].dtype}")
    print(f"   - TRIM(DE.CLAVEPRODUCTO): {df_pedidos['TRIM(DE.CLAVEPRODUCTO)'].dtype}")
    
    # Verificar que las conversiones fueron exitosas
    rutas_numericas = pd.to_numeric(df_pedidos['RUTA'], errors='coerce').notna().sum()
    productos_numericos = pd.to_numeric(df_pedidos['TRIM(DE.CLAVEPRODUCTO)'], errors='coerce').notna().sum()
    total_registros = len(df_pedidos)
    
    print(f"\n✅ RESULTADOS DE CONVERSIÓN:")
    print(f"   - RUTA: {rutas_numericas}/{total_registros} registros convertidos ({(rutas_numericas/total_registros*100):.1f}%)")
    print(f"   - TRIM(DE.CLAVEPRODUCTO): {productos_numericos}/{total_registros} registros convertidos ({(productos_numericos/total_registros*100):.1f}%)")
    
    # Mostrar algunos ejemplos de datos convertidos para verificar
    print(f"\n📝 EJEMPLOS DE DATOS CONVERTIDOS:")
    ejemplos_conversion = df_pedidos[['DAN', 'RUTA', 'TRIM(DE.CLAVEPRODUCTO)', 'Cantidad a enviar']].head(3)
    for _, row in ejemplos_conversion.iterrows():
        print(f"   {row['DAN']} → RUTA: {row['RUTA']} (tipo: {type(row['RUTA'])}) + PRODUCTO: {row['TRIM(DE.CLAVEPRODUCTO)']} (tipo: {type(row['TRIM(DE.CLAVEPRODUCTO)'])}) → Cantidad: {row['Cantidad a enviar']}")
    
    # Verificar si los códigos convertidos existen en el diccionario specs_map
    print(f"\n🔍 VERIFICANDO EXISTENCIA EN DICCIONARIO DE ESPECIFICACIONES:")
    productos_ejemplo = df_pedidos['TRIM(DE.CLAVEPRODUCTO)'].unique()[:5]
    coincidencias_encontradas = 0
    
    for producto in productos_ejemplo:
        encontrado_pallet = (producto, 'Pallet') in specs_map
        encontrado_master = (producto, 'Master') in specs_map
        encontrado_inner = (producto, 'Inner') in specs_map
        encontrado_pieza = (producto, 'Pieza') in specs_map
        
        if any([encontrado_pallet, encontrado_master, encontrado_inner, encontrado_pieza]):
            coincidencias_encontradas += 1
            print(f"   ✅ Producto {producto}: Pallet={encontrado_pallet}, Master={encontrado_master}, Inner={encontrado_inner}, Pieza={encontrado_pieza}")
        else:
            print(f"   ❌ Producto {producto}: No encontrado en diccionario")
    
    print(f"\n📊 RESUMEN DE COINCIDENCIAS:")
    print(f"   - Productos verificados: {len(productos_ejemplo)}")
    print(f"   - Productos encontrados en diccionario: {coincidencias_encontradas}")
    print(f"   - Porcentaje de coincidencias: {(coincidencias_encontradas/len(productos_ejemplo)*100):.1f}%")
    
    if coincidencias_encontradas == 0:
        print(f"\n⚠️ ATENCIÓN: No se encontraron coincidencias en el diccionario de especificaciones")
        print(f"   Esto puede deberse a:")
        print(f"   - Códigos de producto diferentes entre los datos y el diccionario")
        print(f"   - Formato de los códigos en el archivo ZQDIS.xlsx")
        print(f"   - Verificar que la columna SKU en ZQDIS.xlsx coincida con TRIM(DE.CLAVEPRODUCTO)")
        
        # Mostrar algunos ejemplos del diccionario para comparar
        print(f"\n📝 EJEMPLOS DE CÓDIGOS EN EL DICCIONARIO:")
        productos_diccionario = list(set([key[0] for key in specs_map.keys()]))[:5]
        for producto in productos_diccionario:
            print(f"   - {producto} (tipo: {type(producto)})")
    else:
        print(f"\n✅ ¡Excelente! Se encontraron coincidencias. El simulador debería funcionar correctamente.")
    
    # Definir columnas de resultado
    columnas_resultado = [
        'Volumen m3', 'Peso Kg', 'Pallet', 'Master', 'Inner', 'Pieza',
        'Piezas Pallet', 'Volumen Pallet', 'Peso Pallet', 'R1', 'Piezas Master',
        'Volumen Master', 'Peso Master', 'R2', 'Piezas Inner', 'Volumen Inner',
        'Peso Inner', 'R3', 'Piezas Unitario', 'Volumen Unitario', 'Peso Unitario',
        'RF', 'Check'
    ]
    
    try:
        print("\n🔄 Ejecutando cálculos de empaque...")
        resultados = df_pedidos.apply(lambda fila: calcular_empaque_completo(fila, specs_map), axis=1)
        resultados.columns = columnas_resultado
        
        # Combinar datos originales con resultados
        df_final_completo = pd.concat([df_pedidos, resultados], axis=1)
        
        # VOLUMEN: Se mantiene sin conversión adicional para evitar hiperreducción
        print(f"\n🔄 CONVIRTIENDO UNIDADES DE VOLUMEN...")
        print(f"   - Volumen original (promedio): {df_final_completo['Volumen m3'].mean():.6f}")
        
        print(f"   - Volumen final (promedio): {df_final_completo['Volumen m3'].mean():.6f}")
        print(f"   ✅ No se aplica división entre 1,000,000 para evitar doble conversión")
        
        print(f"✅ Cálculos completados exitosamente")
        print(f"   - Registros procesados: {df_final_completo.shape[0]}")
        print(f"   - Columnas totales: {df_final_completo.shape[1]}")
        
        # Mostrar estadísticas de los resultados
        print(f"\n📊 ESTADÍSTICAS DE EMPAQUE:")
        print("-" * 40)
        
        # Estadísticas por tipo de DAN
        if 'DAN' in df_final_completo.columns:
            print("📋 Estadísticas por tipo de DAN:")
            for tipo_dan in df_final_completo['DAN'].unique():
                subset = df_final_completo[df_final_completo['DAN'] == tipo_dan]
                total_volumen = subset['Volumen m3'].sum()
                total_peso = subset['Peso Kg'].sum()
                total_pallets = subset['Pallet'].sum()
                total_cantidad = subset['Cantidad a enviar'].sum()
                
                print(f"   {tipo_dan}:")
                print(f"     - Registros: {len(subset)}")
                print(f"     - Cantidad total: {total_cantidad:,.0f} piezas")
                print(f"     - Volumen total: {total_volumen:.2f} m³")
                print(f"     - Peso total: {total_peso:.2f} kg")
                print(f"     - Pallets requeridos: {total_pallets:,.0f}")
        
        # Estadísticas generales
        print(f"\n📈 TOTALES GENERALES:")
        print(f"   - Volumen total: {df_final_completo['Volumen m3'].sum():.2f} m³")
        print(f"   - Peso total: {df_final_completo['Peso Kg'].sum():.2f} kg")
        print(f"   - Pallets totales: {df_final_completo['Pallet'].sum():,.0f}")
        print(f"   - Masters totales: {df_final_completo['Master'].sum():,.0f}")
        print(f"   - Inners totales: {df_final_completo['Inner'].sum():,.0f}")
        print(f"   - Piezas sueltas: {df_final_completo['Pieza'].sum():,.0f}")
        
        # Identificar productos con warnings
        warnings = df_final_completo[df_final_completo['Check'].str.contains('Aguas', na=False)]
        if not warnings.empty:
            print(f"\n⚠️ PRODUCTOS CON ADVERTENCIAS: {len(warnings)} registros")
            print("   Estos productos tienen residuos no empacables:")
            for _, row in warnings.head(5).iterrows():
                print(f"     - {row['TRIM(DE.CLAVEPRODUCTO)']}: {row['RF']} piezas residuales")
        else:
            print(f"\n✅ Todos los productos se empacaron correctamente")
        
        # Mostrar primeros resultados
        print(f"\n📝 PRIMEROS 5 RESULTADOS:")
        columnas_mostrar = ['DAN', 'RUTA', 'TRIM(DE.CLAVEPRODUCTO)', 'Cantidad a enviar', 
                           'Volumen m3', 'Peso Kg', 'Pallet', 'Master', 'Inner', 'Pieza', 'Check']
        columnas_existentes = [col for col in columnas_mostrar if col in df_final_completo.columns]
        print(df_final_completo[columnas_existentes].head().to_string(index=False))
        
        # CREAR TABLA DINÁMICA: SIMULADOR DANES
        print(f"\n📊 CREANDO TABLA DINÁMICA 'SIMULADOR DANES'...")
        print("-" * 50)
        
        # Crear tabla dinámica con RUTA en filas, DAN en columnas y suma de Volumen m3 en valores
        df_simulador_danes = df_final_completo.pivot_table(
            index='RUTA',           # Filas
            columns='DAN',          # Columnas  
            values='Volumen m3',    # Valores
            aggfunc='sum',          # Función de agregación
            fill_value=0            # Rellenar valores vacíos con 0
        )
        
        # Agregar fila de totales
        df_simulador_danes.loc['TOTAL'] = df_simulador_danes.sum()
        
        # Agregar columna de totales por fila
        df_simulador_danes['TOTAL'] = df_simulador_danes.sum(axis=1)
        
        print(f"✅ Tabla dinámica 'Simulador DANES' creada exitosamente")
        print(f"   - Filas (RUTAS): {df_simulador_danes.shape[0]} (incluyendo total)")
        print(f"   - Columnas (TIPOS DAN): {df_simulador_danes.shape[1]} (incluyendo total)")
        
        # Mostrar información de la tabla dinámica
        print(f"\n📋 ESTRUCTURA DE LA TABLA DINÁMICA:")
        print(f"   - Filas: RUTA ({df_simulador_danes.index.name})")
        print(f"   - Columnas: {list(df_simulador_danes.columns)}")
        print(f"   - Valores: Suma de Volumen m3")
        
        # Mostrar estadísticas de la tabla dinámica
        total_volumen_tabla = df_simulador_danes.loc['TOTAL', 'TOTAL']
        print(f"\n📈 ESTADÍSTICAS DE LA TABLA DINÁMICA:")
        print(f"   - Volumen total: {total_volumen_tabla:.6f} m³")
        print(f"   - Rutas únicas: {len(df_simulador_danes.index) - 1}")  # -1 para excluir la fila TOTAL
        print(f"   - Tipos de DAN: {len(df_simulador_danes.columns) - 1}")  # -1 para excluir la columna TOTAL
        
        # Mostrar la tabla dinámica completa
        print(f"\n📝 TABLA DINÁMICA 'SIMULADOR DANES':")
        print(df_simulador_danes.to_string())
        
        # Crear variable global para la tabla dinámica
        globals()['df_simulador_danes'] = df_simulador_danes
        print(f"\n🔗 DataFrame 'df_simulador_danes' creado para uso posterior")
        
        # Crear variable global para uso posterior
        globals()['df_empaques_calculados'] = df_final_completo
        print(f"\n🔗 DataFrame 'df_empaques_calculados' creado para uso posterior")
        print(f"📄 Los resultados se incluirán automáticamente en el archivo Excel consolidado de la celda 3")
        
    except Exception as e:
        print(f"❌ Error durante el procesamiento: {e}")
        import traceback
        traceback.print_exc()
        
else:
    print("⚠️ No se encontraron datos de DAN para procesar")
    print("   Ejecuta primero las celdas anteriores para generar df_danes_total_final")
    print("\n📋 CONFIGURACIÓN MANUAL:")
    print("   Para procesar un archivo externo, usa:")
    print("   archivo_pedidos = 'ruta_a_tu_archivo.xlsx'")
    print("   df_pedidos = pd.read_excel(archivo_pedidos)")
    print("   # Luego aplica calcular_empaque_completo()")

print(f"\n🎉 SIMULADOR DE EMPAQUES COMPLETADO!")
print("="*60)

🔧 INICIANDO SIMULADOR DE EMPAQUES...
📖 Leyendo archivo de especificaciones...
✅ Archivo leído correctamente: 91470 registros encontrados
✅ Diccionario de especificaciones creado: 91469 entradas únicas

📝 Ejemplos del diccionario de especificaciones:
   (10000, 'Pieza') → {'Piezas': 1, 'Volumen': 0.006928852, 'Peso': 0.001653}
   (10001, 'Pieza') → {'Piezas': 1, 'Volumen': 0.000181125, 'Peso': 5.8e-05}
   (10002, 'Pieza') → {'Piezas': 1, 'Volumen': 0.00103635, 'Peso': 0.000208}

🧪 PRUEBA RÁPIDA DEL SIMULADOR:
----------------------------------------
📦 Usando SKU de ejemplo: 10000
🎯 Resultado para 1250 piezas:
   Pallets: 3
   Masters: 0
   Inners: 20
   Piezas: 2
   Volumen total: 11.11 m³
   Peso total: 2.22 kg
   Estado: ✅ OK

🚀 PROCESANDO DATOS DE DAN CON SIMULADOR DE EMPAQUES...
------------------------------------------------------------
✅ Usando df_danes_total_final: 283 registros

🔢 CONVIRTIENDO DATOS A FORMATO NUMÉRICO PARA BÚSQUEDAS...
------------------------------------------

05. Simulador de Carga

In [5]:
# ==================== SIMULADOR DE CARGA ====================
import pandas as pd
import pprint
import os
from datetime import datetime

print("🚛 INICIANDO SIMULADOR DE CARGA...")
print("="*60)

# Verificar que exista df_pc_agrupado
if 'df_pc_agrupado' in locals() and df_pc_agrupado is not None:
    print(f"✅ DataFrame df_pc_agrupado encontrado: {df_pc_agrupado.shape[0]} registros")
    
    # Verificar que exista el diccionario de especificaciones
    if 'specs_map' in globals() and specs_map is not None:
        print(f"✅ Diccionario de especificaciones encontrado: {len(specs_map)} entradas")
        
        # --- PREPARACIÓN DE DATOS ---
        print(f"\n📋 PREPARANDO DATOS DE PC PARA SIMULACIÓN...")
        print("-" * 50)
        
        # Crear df_pedidos basado en df_pc_agrupado (igual que el simulador de DANES)
        # Mapear las columnas según los requerimientos
        df_pedidos = df_pc_agrupado[['RUTA', 'TRIM(DE.CLAVEPRODUCTO)', 'Suma de CANTIDADCARGADA']].copy()
        
        # Renombrar columnas para que coincidan con el simulador original
        df_pedidos = df_pedidos.rename(columns={
            'Suma de CANTIDADCARGADA': 'Cantidad a enviar'
        })
        
        # Limpiar la columna de código de producto
        df_pedidos['TRIM(DE.CLAVEPRODUCTO)'] = df_pedidos['TRIM(DE.CLAVEPRODUCTO)'].astype(str).str.strip()
        
        print(f"✅ Datos preparados:")
        print(f"   - Registros: {df_pedidos.shape[0]}")
        print(f"   - Columnas: {list(df_pedidos.columns)}")
        print(f"   - Total cantidad a enviar: {df_pedidos['Cantidad a enviar'].sum():,.0f}")
        
        # --- CONVERSIÓN A FORMATO NUMÉRICO ---
        print(f"\n🔢 CONVIRTIENDO DATOS A FORMATO NUMÉRICO...")
        df_pedidos['TRIM(DE.CLAVEPRODUCTO)'] = pd.to_numeric(df_pedidos['TRIM(DE.CLAVEPRODUCTO)'], errors='coerce')
        productos_numericos = df_pedidos['TRIM(DE.CLAVEPRODUCTO)'].notna().sum()
        print(f"   - Productos convertidos exitosamente: {productos_numericos}/{len(df_pedidos)}")
        
        # --- VERIFICAR COINCIDENCIAS ---
        print(f"\n🔍 VERIFICANDO COINCIDENCIAS CON DICCIONARIO...")
        productos_ejemplo = df_pedidos['TRIM(DE.CLAVEPRODUCTO)'].dropna().unique()[:10]
        coincidencias_encontradas = 0
        
        for producto in productos_ejemplo:
            encontrado_pallet = (producto, 'Pallet') in specs_map
            encontrado_master = (producto, 'Master') in specs_map
            encontrado_inner = (producto, 'Inner') in specs_map
            encontrado_pieza = (producto, 'Pieza') in specs_map
            
            if any([encontrado_pallet, encontrado_master, encontrado_inner, encontrado_pieza]):
                coincidencias_encontradas += 1
        
        print(f"   - Productos verificados: {len(productos_ejemplo)}")
        print(f"   - Productos encontrados: {coincidencias_encontradas}")
        print(f"   - Porcentaje coincidencias: {(coincidencias_encontradas/len(productos_ejemplo)*100):.1f}%")
        
        # --- APLICAR SIMULADOR DE EMPAQUES ---
        print(f"\n🔧 APLICANDO SIMULADOR DE EMPAQUES...")
        print("-" * 60)
        
        # Definir columnas de resultado (igual que simulador DANES)
        columnas_resultado = [
            'Volumen m3', 'Peso Kg', 'Pallet', 'Master', 'Inner', 'Pieza',
            'Piezas Pallet', 'Volumen Pallet', 'Peso Pallet', 'R1', 'Piezas Master',
            'Volumen Master', 'Peso Master', 'R2', 'Piezas Inner', 'Volumen Inner',
            'Peso Inner', 'R3', 'Piezas Unitario', 'Volumen Unitario', 'Peso Unitario',
            'RF', 'Check'
        ]
        
        # Usar la misma función calcular_empaque_completo que ya existe
        if 'calcular_empaque_completo' in globals():
            try:
                print("🔄 Ejecutando cálculos de empaque...")
                resultados = df_pedidos.apply(lambda fila: calcular_empaque_completo(fila, specs_map), axis=1)
                resultados.columns = columnas_resultado
                
                # Combinar datos originales con resultados
                df_final_completo = pd.concat([df_pedidos, resultados], axis=1)
                
                # VOLUMEN: Se mantiene sin conversión adicional para evitar hiperreducción
                print(f"🔄 REVISANDO UNIDADES DE VOLUMEN...")
                print(f"   ✅ No se aplica división entre 1,000,000 para evitar doble conversión")
                
                print(f"✅ Cálculos completados exitosamente")
                print(f"   - Registros procesados: {df_final_completo.shape[0]}")
                print(f"   - Columnas totales: {df_final_completo.shape[1]}")
                
                # Mostrar estadísticas del resultado
                print(f"\n📊 ESTADÍSTICAS DEL RESULTADO:")
                print(f"   - Volumen total: {df_final_completo['Volumen m3'].sum():.2f} m³")
                print(f"   - Peso total: {df_final_completo['Peso Kg'].sum():.2f} kg")
                print(f"   - Pallets totales: {df_final_completo['Pallet'].sum():,.0f}")
                print(f"   - Masters totales: {df_final_completo['Master'].sum():,.0f}")
                print(f"   - Inners totales: {df_final_completo['Inner'].sum():,.0f}")
                print(f"   - Piezas sueltas: {df_final_completo['Pieza'].sum():,.0f}")
                
                # Mostrar primeros resultados
                print(f"\n📝 PRIMEROS 5 RESULTADOS:")
                columnas_mostrar = ['RUTA', 'TRIM(DE.CLAVEPRODUCTO)', 'Cantidad a enviar', 
                                   'Volumen m3', 'Peso Kg', 'Pallet', 'Master', 'Inner', 'Pieza', 'Check']
                print(df_final_completo[columnas_mostrar].head().to_string(index=False))
                
                # --- CREAR TABLA DINÁMICA ---
                print(f"\n📊 CREANDO TABLA DINÁMICA 'SIMULADOR CARGA'...")
                print("-" * 50)
                
                # Crear tabla dinámica simple: solo RUTA en filas y Suma de Volumen m3 en valores
                df_simulador_carga = df_final_completo.groupby('RUTA').agg({
                    'Volumen m3': 'sum'
                }).rename(columns={'Volumen m3': 'Suma de Volumen m3'})
                
                # Agregar fila de totales
                df_simulador_carga.loc['TOTAL'] = df_simulador_carga.sum()
                
                print(f"✅ Tabla dinámica 'Simulador CARGA' creada exitosamente")
                print(f"   - Filas (RUTAS): {df_simulador_carga.shape[0]} (incluyendo total)")
                print(f"   - Columnas: {list(df_simulador_carga.columns)}")
                
                # Mostrar estadísticas de la tabla dinámica
                total_volumen_tabla = df_simulador_carga.loc['TOTAL', 'Suma de Volumen m3']
                print(f"\n📈 ESTADÍSTICAS DE LA TABLA DINÁMICA:")
                print(f"   - Volumen total: {total_volumen_tabla:.6f} m³")
                print(f"   - Rutas únicas: {len(df_simulador_carga.index) - 1}")
                
                # Mostrar la tabla dinámica completa
                print(f"\n📝 TABLA DINÁMICA 'SIMULADOR CARGA':")
                print(df_simulador_carga.to_string())
                
                # Crear variables globales para uso posterior
                globals()['df_simulador_carga'] = df_simulador_carga
                globals()['df_empaques_calculados_carga'] = df_final_completo
                
                print(f"\n🔗 DataFrames creados para uso posterior:")
                print(f"   - df_empaques_calculados_carga: {df_final_completo.shape[0]} registros")
                print(f"   - df_simulador_carga: {df_simulador_carga.shape[0]} rutas")
                print(f"📄 Los resultados se incluirán automáticamente en el archivo Excel consolidado")
                
            except Exception as e:
                print(f"❌ Error durante el procesamiento: {e}")
                import traceback
                traceback.print_exc()
                
        else:
            print("❌ No se encontró la función calcular_empaque_completo")
            print("   Ejecuta primero la celda del Simulador de DANES")
            
    else:
        print("❌ No se encontró el diccionario de especificaciones (specs_map)")
        print("   Ejecuta primero la celda del Simulador de DANES")
        
else:
    print("❌ No se encontró df_pc_agrupado")
    print("   Ejecuta primero la celda de Calculo PC")

print(f"\n🎉 SIMULADOR DE CARGA COMPLETADO!")
print("="*60)

🚛 INICIANDO SIMULADOR DE CARGA...
✅ DataFrame df_pc_agrupado encontrado: 85728 registros
✅ Diccionario de especificaciones encontrado: 91469 entradas

📋 PREPARANDO DATOS DE PC PARA SIMULACIÓN...
--------------------------------------------------
✅ Datos preparados:
   - Registros: 85728
   - Columnas: ['RUTA', 'TRIM(DE.CLAVEPRODUCTO)', 'Cantidad a enviar']
   - Total cantidad a enviar: 15,325,251

🔢 CONVIRTIENDO DATOS A FORMATO NUMÉRICO...
   - Productos convertidos exitosamente: 85660/85728

🔍 VERIFICANDO COINCIDENCIAS CON DICCIONARIO...
   - Productos verificados: 10
   - Productos encontrados: 10
   - Porcentaje coincidencias: 100.0%

🔧 APLICANDO SIMULADOR DE EMPAQUES...
------------------------------------------------------------
🔄 Ejecutando cálculos de empaque...
🔄 REVISANDO UNIDADES DE VOLUMEN...
   ✅ No se aplica división entre 1,000,000 para evitar doble conversión
✅ Cálculos completados exitosamente
   - Registros procesados: 85728
   - Columnas totales: 26

📊 ESTADÍSTICAS DE

06. Exportación Excel Consolidado

In [6]:
# ==================== EXPORTACIÓN EXCEL CONSOLIDADO ====================
print("📄 EXPORTANDO ARCHIVO EXCEL CONSOLIDADO...")
print("="*60)

# Verificar que existan todos los DataFrames necesarios
print(f"\n🔍 VERIFICANDO DATAFRAMES DISPONIBLES:")

# DataFrames básicos (de celdas 1, 2, 3)
dataframes_basicos = {
    'df_danes_agrupado': 'Calculo DAN',
    'df_pc_agrupado': 'Calculo PC', 
    'df_materiales': 'Esc y Calent',
    'df_danes_total': 'Calculo DAN Total'
}

# DataFrames del simulador (de celdas 4 y 5)
dataframes_simulador = {
    'df_empaques_calculados': 'Simulacion Empaques',
    'df_simulador_danes': 'Simulador DANES',
    'df_empaques_calculados_carga': 'Simulacion Carga',
    'df_simulador_carga': 'Simulador Carga'
}

# Verificar DataFrames básicos
disponibles_basicos = {}
for var_name, hoja_name in dataframes_basicos.items():
    if var_name in locals() and locals()[var_name] is not None:
        disponibles_basicos[var_name] = hoja_name
        print(f"   ✅ {hoja_name}: {locals()[var_name].shape[0]} registros")
    else:
        print(f"   ❌ {hoja_name}: No encontrado en locals()")

# Verificar DataFrames del simulador
disponibles_simulador = {}
for var_name, hoja_name in dataframes_simulador.items():
    if var_name in globals() and globals()[var_name] is not None:
        disponibles_simulador[var_name] = hoja_name
        df_temp = globals()[var_name]
        if var_name in ['df_simulador_danes', 'df_simulador_carga']:
            print(f"   ✅ {hoja_name}: {df_temp.shape[0]} filas x {df_temp.shape[1]} columnas (tabla dinámica)")
        else:
            print(f"   ✅ {hoja_name}: {df_temp.shape[0]} registros")
    else:
        print(f"   ❌ {hoja_name}: No encontrado en globals()")

# Continuar solo si hay DataFrames disponibles
total_disponibles = len(disponibles_basicos) + len(disponibles_simulador)

if total_disponibles > 0:
    # EXPORTACIÓN CONSOLIDADA
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    nombre_output = f'Reporte_Consolidado_DAN_PC_{timestamp}.xlsx'
    ruta_output = os.path.join(carpeta_outputs, nombre_output)
    
    print(f"\n📊 CREANDO ARCHIVO EXCEL CONSOLIDADO:")
    print(f"   📁 Nombre: {nombre_output}")
    print(f"   📂 Ubicación: {carpeta_outputs}")
    
    try:
        with pd.ExcelWriter(ruta_output, engine='openpyxl') as writer:
            hojas_creadas = 0
            
            # Exportar DataFrames básicos
            for var_name, hoja_name in disponibles_basicos.items():
                try:
                    df = locals()[var_name]
                    df.to_excel(writer, sheet_name=hoja_name, index=False)
                    hojas_creadas += 1
                    print(f"   ✅ {hoja_name}: Exportado exitosamente")
                except Exception as e:
                    print(f"   ❌ Error exportando {hoja_name}: {e}")
            
            # Exportar DataFrames del simulador
            for var_name, hoja_name in disponibles_simulador.items():
                try:
                    df = globals()[var_name]
                    # Para tablas dinámicas, incluir índice
                    incluir_indice = (var_name in ['df_simulador_danes', 'df_simulador_carga'])
                    df.to_excel(writer, sheet_name=hoja_name, index=incluir_indice)
                    hojas_creadas += 1
                    print(f"   ✅ {hoja_name}: Exportado exitosamente")
                except Exception as e:
                    print(f"   ❌ Error exportando {hoja_name}: {e}")
        
        # Mensaje de éxito
        print(f"\n💾 ARCHIVO CONSOLIDADO EXPORTADO EXITOSAMENTE!")
        print(f"📁 Ubicación: {ruta_output}")
        print(f"📄 Nombre: {nombre_output}")
        print(f"📋 Hojas creadas: {hojas_creadas}")
        
        # Resumen detallado de hojas
        print(f"\n📊 RESUMEN DE HOJAS EXPORTADAS:")
        for var_name, hoja_name in {**disponibles_basicos, **disponibles_simulador}.items():
            if var_name in locals():
                df = locals()[var_name]
            else:
                df = globals()[var_name]
            
            if var_name in ['df_simulador_danes', 'df_simulador_carga']:
                print(f"   - {hoja_name}: {df.shape[0]} filas x {df.shape[1]} columnas (tabla dinámica)")
            else:
                print(f"   - {hoja_name}: {df.shape[0]} registros")
        
        print(f"\n🎯 ORDEN DE EJECUCIÓN RECOMENDADO:")
        print(f"   1. Celda 1: Calculo DAN con Reporte DANES")
        print(f"   2. Celda 2: Calculo DAN con PC") 
        print(f"   3. Celda 3: Calculo DAN Total")
        print(f"   4. Celda 4: Simulador de DANES")
        print(f"   5. Celda 5: Simulador de Carga")
        print(f"   6. Esta celda: Exportación Excel Consolidado")
        
    except Exception as e:
        print(f"❌ ERROR AL EXPORTAR ARCHIVO CONSOLIDADO:")
        print(f"   {e}")
        import traceback
        traceback.print_exc()
        
else:
    print(f"\n❌ NO SE PUEDEN EXPORTAR DATOS:")
    print(f"   No se encontraron DataFrames disponibles para exportar")
    print(f"   Ejecuta las celdas anteriores en orden para generar los datos")
    
    print(f"\n🔧 SOLUCIÓN:")
    print(f"   1. Ejecuta las celdas 1, 2, 3 y 4 en orden")
    print(f"   2. Luego ejecuta esta celda de exportación")

print(f"\n🎉 EXPORTACIÓN COMPLETADA!")
print("="*60)

📄 EXPORTANDO ARCHIVO EXCEL CONSOLIDADO...

🔍 VERIFICANDO DATAFRAMES DISPONIBLES:
   ✅ Calculo DAN: 259 registros
   ✅ Calculo PC: 85728 registros
   ✅ Esc y Calent: 288 registros
   ✅ Calculo DAN Total: 283 registros
   ✅ Simulacion Empaques: 283 registros
   ✅ Simulador DANES: 129 filas x 4 columnas (tabla dinámica)
   ✅ Simulacion Carga: 85728 registros
   ✅ Simulador Carga: 289 filas x 1 columnas (tabla dinámica)

📊 CREANDO ARCHIVO EXCEL CONSOLIDADO:
   📁 Nombre: Reporte_Consolidado_DAN_PC_20260511_105136.xlsx
   📂 Ubicación: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Calculo de PC y DAN\Abastos\Outputs
   ✅ Calculo DAN: Exportado exitosamente
   ✅ Calculo PC: Exportado exitosamente
   ✅ Esc y Calent: Exportado exitosamente
   ✅ Calculo DAN Total: Exportado exitosamente
   ✅ Simulacion Empaques: Exportado exitosamente
   ✅ Simulador DANES: Exportado exitosamente
   ✅ Simulacion Carga: Exportado exitosamente
   ✅ Simulador Carga: Exportado exitosamente

💾 A